# Assignment 2 — Transformer language models (DL4NLP, 2026)

Part 0, 1, and 2 are reused from assignment 1 (modified in case of need). Part 3 is specific to AS2.

## Part 0 — Setup

In [1]:
# Part 0.1

# print Python path to make sure VS Code uses my course environment/kernel
import sys
print("python:", sys.executable)

import os
import random
from pathlib import Path

try:
    import torch
    import nltk
    import datasets
    import transformers
except ModuleNotFoundError as e:
    raise RuntimeError(
        "Some required package is missing. Install with:\n"
        "python -m pip install torch nltk datasets transformers matplotlib scikit-learn"
    ) from e

# NLTK tokenizer files
nltk.download("punkt", quiet=True)

try:
    nltk.download("punkt_tab", quiet=True)
except Exception:
    pass

# Fix random seed so results are more reproducible
SEED = 0
random.seed(SEED)
torch.manual_seed(SEED)

# Device selection:
# cuda = NVIDIA GPU, mps = Apple Silicon GPU, else CPU
has_mps = hasattr(torch.backends, "mps") and torch.backends.mps.is_available()
DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if has_mps else "cpu")
print("device:", DEVICE)

# print versions for report/debugging
print("torch:", torch.__version__)
print("datasets:", datasets.__version__)
print("transformers:", transformers.__version__)

python: /Users/ANOURI/Desktop/accepted PhD/courses/2026 Spring/LLMWASP/.venv/bin/python


/Users/ANOURI/Desktop/accepted PhD/courses/2026 Spring/LLMWASP/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device: mps
torch: 2.12.0
datasets: 4.8.5
transformers: 5.10.2


In [2]:
# Part 0.1 - Download and extract archives

import zipfile
import urllib.request
from pathlib import Path


ROOT_DIR = Path(".").resolve()
ZIP_URL = "https://www.cse.chalmers.se/~richajo/waspnlp2026/a1_1.zip"
ZIP_PATH = ROOT_DIR / "a1_1.zip"

# Only download if not already present locally
if not ZIP_PATH.exists():
    print("Downloading assignment 1 data archive...")
    urllib.request.urlretrieve(ZIP_URL, ZIP_PATH)  # fetch and save the zip file
    print("Downloaded:", ZIP_PATH)
else:
    print("Zip already exists:", ZIP_PATH)

# Extract everything into ROOT
with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    zf.extractall(ROOT_DIR)
print("a1_1 archive extracted under:", ROOT_DIR)

# Path to the extracted data folder and the two text files we need.
DATA_DIR = ROOT_DIR / "a1_1"
TRAIN_FILE_PATH = (DATA_DIR / "train.txt").resolve()
VAL_FILE_PATH = (DATA_DIR / "val.txt").resolve()

# Add aliases used later in the notebook
TRAIN_FILE = str(TRAIN_FILE_PATH)
VAL_FILE = str(VAL_FILE_PATH) if VAL_FILE_PATH.exists() else None

# Quick existence checks so I notice immediately if extraction failed,
# instead of getting a confusing error much later in the notebook.
print("train.txt found?", TRAIN_FILE_PATH.exists())
print("val.txt found?  ", VAL_FILE_PATH.exists())
print("TRAIN_FILE_PATH =", TRAIN_FILE_PATH)
print("VAL_FILE_PATH   =", VAL_FILE_PATH)


# a1_2.zip: NEW for Assignment 2
A2_ZIP_URL = "https://www.cse.chalmers.se/~richajo/waspnlp2026/a1_2.zip"
A2_ZIP_PATH = ROOT_DIR / "a1_2.zip"

# Same "download only if missing" pattern as above, kept consistent.
if not A2_ZIP_PATH.exists():
    print("Downloading assignment 2 skeleton archive...")
    urllib.request.urlretrieve(A2_ZIP_URL, A2_ZIP_PATH)
    print("Downloaded:", A2_ZIP_PATH)
else:
    print("Zip already exists:", A2_ZIP_PATH)

# Extract the skeleton
with zipfile.ZipFile(A2_ZIP_PATH, "r") as zf:
    zf.extractall(ROOT_DIR)
    print("a1_2 skeleton contents:", zf.namelist())

# Folder where the skeleton was extracted
A2_DIR = ROOT_DIR / "a1_2"
print("a1_2 skeleton extracted under:", A2_DIR)

Zip already exists: /Users/ANOURI/Desktop/accepted PhD/courses/2026 Spring/LLMWASP/AssignmentsFinal/a1_1.zip
a1_1 archive extracted under: /Users/ANOURI/Desktop/accepted PhD/courses/2026 Spring/LLMWASP/AssignmentsFinal
train.txt found? True
val.txt found?   True
TRAIN_FILE_PATH = /Users/ANOURI/Desktop/accepted PhD/courses/2026 Spring/LLMWASP/AssignmentsFinal/a1_1/train.txt
VAL_FILE_PATH   = /Users/ANOURI/Desktop/accepted PhD/courses/2026 Spring/LLMWASP/AssignmentsFinal/a1_1/val.txt
Zip already exists: /Users/ANOURI/Desktop/accepted PhD/courses/2026 Spring/LLMWASP/AssignmentsFinal/a1_2.zip
a1_2 skeleton contents: ['a1_2/', 'a1_2/A2_skeleton.py']
a1_2 skeleton extracted under: /Users/ANOURI/Desktop/accepted PhD/courses/2026 Spring/LLMWASP/AssignmentsFinal/a1_2


In [3]:
# Sanity check: list zip contents once, useful only for debugging if something looks wrong.

import zipfile

# a1_1.zip contents (data)
print("=== a1_1.zip contents ===")
with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    names = zf.namelist()
    print("Zip entries:", len(names))
    for n in names:
        print(" -", n)

# a1_2.zip contents (Assignment 2 skeleton)
print("\n=== a1_2.zip contents ===")
with zipfile.ZipFile(A2_ZIP_PATH, "r") as zf:
    names = zf.namelist()
    print("Zip entries:", len(names))
    for n in names:
        print(" -", n)

=== a1_1.zip contents ===
Zip entries: 4
 - a1_1/
 - a1_1/train.txt
 - a1_1/A1_skeleton.py
 - a1_1/val.txt

=== a1_2.zip contents ===
Zip entries: 2
 - a1_2/
 - a1_2/A2_skeleton.py


In [4]:
import ast

with open(A2_DIR / "A2_skeleton.py", encoding="utf-8") as f:
    tree = ast.parse(f.read())

for node in tree.body:
    if isinstance(node, (ast.ClassDef, ast.FunctionDef)):
        print(f"{'class' if isinstance(node, ast.ClassDef) else 'def'} {node.name}")
        for sub in node.body:
            if isinstance(sub, ast.FunctionDef):
                print(f"    def {sub.name}({', '.join(a.arg for a in sub.args.args)})")

class A2ModelConfig
    def __init__(self, vocab_size, hidden_size, intermediate_size, num_attention_heads, num_hidden_layers, rope_theta, hidden_act, max_position_embeddings, rms_norm_eps)
class A2MLP
    def __init__(self, config)
    def forward(self, hidden_states)
class A2RMSNorm
    def __init__(self, config)
    def forward(self, hidden_states)
class A2Attention
    def __init__(self, config)
    def forward(self, hidden_states, rope_rotations)
class A2DecoderLayer
    def __init__(self, config)
    def forward(self, hidden_states, rope_rotations)
class A2Transformer
    def __init__(self, config)
    def forward(self, input_ids, labels)
def apply_rotary_pos_emb
def rotate_half
class A2RotaryEmbedding
    def __init__(self, config, device)
    def forward(self, x)


In [5]:
with open(A2_DIR / "A2_skeleton.py", encoding="utf-8") as f:
    lines = f.readlines()

print(f"Total lines: {len(lines)}\n")
chunk_size = 60
for i in range(0, len(lines), chunk_size):
    print(f"--- lines {i+1}-{min(i+chunk_size, len(lines))} ---")
    print("".join(lines[i:i+chunk_size]))

Total lines: 138

--- lines 1-60 ---

import torch
from torch import nn
from transformers import PreTrainedModel, PretrainedConfig


class A2ModelConfig(PretrainedConfig):
    """Configuration object that stores hyperparameters that define the Transformer language model."""
    def __init__(self, vocab_size=None, hidden_size=None, intermediate_size=None, num_attention_heads=None, 
                 num_hidden_layers=None,
                 rope_theta=None, hidden_act='silu', max_position_embeddings=None, rms_norm_eps=None, **kwargs):
        super().__init__(**kwargs)
        self.vocab_size = vocab_size
        self.hidden_size = hidden_size
        self.max_position_embeddings = max_position_embeddings
        self.rms_norm_eps = rms_norm_eps
        self.num_attention_heads = num_attention_heads
        self.rope_theta = rope_theta
        self.hidden_act = hidden_act
        self.intermediate_size = intermediate_size
        self.num_hidden_layers = num_hidden_layers



class A2MLP(n

## Part 1 — Tokenization

### Task 1.1 — Word splitting with NLTK

In [6]:
# Task 1.1 - I need nltk's word_tokenize for word splitting.
from nltk.tokenize import word_tokenize

def split_words(text: str) -> list[str]:
    """Lowercase + NLTK word tokenization."""
    # lowercase first so 'The' and 'the' become the same token later.
    return word_tokenize(text.lower(), language='english')

# Sanity check (from the assignment hint)
print(split_words("April (Apr.) is the fourth month of the year."))

['april', '(', 'apr', '.', ')', 'is', 'the', 'fourth', 'month', 'of', 'the', 'year', '.']


In [9]:
# Task 1.2 - Building the vocabulary

# Special symbols required by the assignment:
# - PAD: padding so batches can be rectangular
# - BOS: beginning of paragraph
# - EOS: end of paragraph
# - UNK: unseen / low-frequency words
PAD_TOK = '<pad>'
BOS_TOK = '<bos>'
EOS_TOK = '<eos>'
UNK_TOK = '<unk>'
SPECIALS = [PAD_TOK, BOS_TOK, EOS_TOK, UNK_TOK]

def build_vocabulary(paragraphs, max_voc_size: int):
    # Counter lets me count word frequencies.
    counter = Counter()
    for p in paragraphs:
        counter.update(split_words(p))

    # reserve slots for the 4 special symbols, then fill the rest
    # with the most frequent words
    n_keep = max(0, max_voc_size - len(SPECIALS))
    most_common = [w for w, _ in counter.most_common(n_keep)]

    # itos: int -> string (used to go back from ids to words)
    # stoi: string -> int (used to encode text into ids)
    itos = list(SPECIALS) + most_common
    stoi = {w: i for i, w in enumerate(itos)}
    return stoi, itos

In [10]:
from datasets import load_dataset
from collections import Counter

# Build dataset first, then build vocabulary from TRAIN ONLY
if VAL_FILE and os.path.exists(VAL_FILE):
    raw = load_dataset('text', data_files={'train': TRAIN_FILE, 'val': VAL_FILE})
    # drop empty lines.
    raw = raw.filter(lambda x: x['text'].strip() != '')
    dataset = {'train': raw['train'], 'val': raw['val']}
else:
    raw = load_dataset('text', data_files={'train': TRAIN_FILE})
    raw = raw.filter(lambda x: x['text'].strip() != '')
    split = raw['train'].train_test_split(test_size=0.11, seed=SEED)
    dataset = {'train': split['train'], 'val': split['test']}
    print('No official validation text file found -> using deterministic train/val split (89/11).')

print('dataset sizes:', {k: len(v) for k, v in dataset.items()})

# extract just the text strings, since build_vocabulary expects raw paragraphs.
train_paragraphs = [dataset['train'][i]['text'] for i in range(len(dataset['train']))]
print('train paragraphs used for vocab:', len(train_paragraphs))

MAX_VOC_SIZE = 10_000
stoi, itos = build_vocabulary(train_paragraphs, MAX_VOC_SIZE)
print('vocab size:', len(itos))

# Sanity checks
assert len(itos) <= MAX_VOC_SIZE
for s in SPECIALS:
    assert s in stoi
for w in ['the', 'and']:
    assert w in stoi, f'expected common word {w!r} to be in vocab'
for w in ['cuboidal', 'epiglottis']:
    print(f'{w!r} in vocab? {w in stoi}')

# Round-trip check: word > id > word should give back the same word.
t = 'language'
assert itos[stoi[t]] == t
print('Sanity checks passed.')

Generating train split: 294118 examples [00:00, 669984.84 examples/s]
Generating val split: 35748 examples [00:00, 713718.49 examples/s]
Filter: 100%|██████████| 35748/35748 [00:00<00:00, 904554.08 examples/s]


dataset sizes: {'train': 147059, 'val': 17874}
train paragraphs used for vocab: 147059
vocab size: 10000
'cuboidal' in vocab? False
'epiglottis' in vocab? False
Sanity checks passed.


In [11]:
# Task 1.3 - A1Tokenizer (HuggingFace-like)

class A1Tokenizer:
    """Minimal HuggingFace-style tokenizer.

    Wraps the (stoi, itos) vocabulary + NLTK word splitting,
    and turns text into input_ids (+ attention_mask), like HF tokenizers do.
    """

    pad_token = PAD_TOK
    bos_token = BOS_TOK
    eos_token = EOS_TOK
    unk_token = UNK_TOK

    def __init__(self, stoi: dict[str, int], itos: list[str], max_length: int = 256):
        # __init__ just stores the vocabulary and settings I'll need later.
        self.stoi = stoi
        self.itos = itos
        self.max_length = max_length
        self.pad_token_id = stoi[PAD_TOK]
        self.bos_token_id = stoi[BOS_TOK]
        self.eos_token_id = stoi[EOS_TOK]
        self.unk_token_id = stoi[UNK_TOK]

    def __len__(self):
        # __len__ should return vocab size, per assignment instructions.
        return len(self.itos)

    def _encode_one(self, text: str) -> list[int]:
        # wrap every paragraph with BOS/EOS symbols, and map unseen words
        # to UNK using .get() with a default, instead of crashing on KeyError.
        ids = [self.bos_token_id]
        for w in split_words(text):
            ids.append(self.stoi.get(w, self.unk_token_id))
        ids.append(self.eos_token_id)
        return ids

    def decode(self, ids):
        # Turns ids back into words, used later for inspecting model output.
        return [self.itos[i] for i in ids]

    def __call__(self, texts, return_tensors=None, padding=False, truncation=False, max_length=None):
        # __call__ makes tokenizer(texts) work like a function call
        # (this is what HuggingFace tokenizers do too).
        if isinstance(texts, str):
            texts = [texts]
        max_length = max_length or self.max_length
        encoded = [self._encode_one(t) for t in texts]

        if truncation:
            truncated = []
            for ids in encoded:
                if len(ids) > max_length:
                    ids = ids[:max_length]
                    # I force the last token to stay EOS after cutting,
                    # so the paragraph still "looks" properly closed.
                    ids[-1] = self.eos_token_id
                truncated.append(ids)
            encoded = truncated

        if padding:
            # pad every sequence to the length of the longest one in the batch,
            # so I can stack them into one rectangular tensor.
            L = max(len(ids) for ids in encoded)
            attn = [[1] * len(ids) + [0] * (L - len(ids)) for ids in encoded]
            encoded = [ids + [self.pad_token_id] * (L - len(ids)) for ids in encoded]
        else:
            attn = [[1] * len(ids) for ids in encoded]

        out = {'input_ids': encoded, 'attention_mask': attn}
        if return_tensors == 'pt':
            # Convert plain Python lists into torch tensors when requested.
            out = {k: torch.tensor(v, dtype=torch.long) for k, v in out.items()}
        return out

    def save(self, path: str):
        # only need to save itos (stoi can be rebuilt from it) + max_length.
        with open(path, 'w', encoding='utf-8') as f:
            json.dump({'itos': self.itos, 'max_length': self.max_length}, f)

    @classmethod
    def from_file(cls, path: str):
        # Reload tokenizer without rebuilding the vocab from scratch.
        with open(path, encoding='utf-8') as f:
            obj = json.load(f)
        itos = obj['itos']
        stoi = {w: i for i, w in enumerate(itos)}
        return cls(stoi, itos, max_length=obj.get('max_length', 256))


def build_tokenizer(paragraphs, max_voc_size: int, max_length: int = 256) -> A1Tokenizer:
    # Convenience function: build vocab + create tokenizer in one step.
    stoi, itos = build_vocabulary(paragraphs, max_voc_size)
    return A1Tokenizer(stoi, itos, max_length=max_length)

In [12]:
# Task 1.3 - Build tokenizer

import json  # needed by tokenizer save/load methods

# 1) Build tokenizer from training paragraphs and vocabulary limit.
tokenizer = build_tokenizer(train_paragraphs, MAX_VOC_SIZE)

# 2) Quick sanity check (optional but useful): test padding + truncation behavior.
test_texts = ["This is a test.", "Another test."]
out = tokenizer(test_texts, return_tensors="pt", padding=True, truncation=True)
print(out)

# 3) Decode first sample so I can visually inspect BOS/words/EOS.
print("decoded[0]:", tokenizer.decode(out["input_ids"][0].tolist()))

# 4) Save and reload tokenizer (optional, but useful for RELOAD_FROM_DISK workflow later).
tokenizer.save("a1_tokenizer.json")
tok2 = A1Tokenizer.from_file("a1_tokenizer.json")
assert len(tok2) == len(tokenizer)
print("Saved & reloaded OK. |V| =", len(tok2))

# 5) Verify BOS/EOS placement for each sequence.
for row, mask in zip(out["input_ids"].tolist(), out["attention_mask"].tolist()):
    n_real = sum(mask)                      # number of non-padding tokens
    assert row[0] == tokenizer.bos_token_id
    assert row[n_real - 1] == tokenizer.eos_token_id

{'input_ids': tensor([[  1,  35,  14,  11, 972,   6,   2],
        [  1, 155, 972,   6,   2,   0,   0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 0, 0]])}
decoded[0]: ['<bos>', 'this', 'is', 'a', 'test', '.', '<eos>']
Saved & reloaded OK. |V| = 10000


## Part 2: Loading the text files and creating batches

In [13]:
# Task 2.1 - Loading the texts with HuggingFace Datasets

from datasets import load_dataset

# load train/val directly from the text files (one paragraph per line)
dataset = load_dataset('text', data_files={'train': TRAIN_FILE, 'val': VAL_FILE})

# remove empty lines, as instructed.
dataset = dataset.filter(lambda x: x['text'].strip() != '')

print('train size:', len(dataset['train']))
print('val size:  ', len(dataset['val']))

# Sanity check from the assignment: ~147,000 train / ~18,000 val instances
assert 140_000 <= len(dataset['train']) <= 155_000, \
    f"unexpected train size: {len(dataset['train'])}"
assert 15_000 <= len(dataset['val']) <= 21_000, \
    f"unexpected val size: {len(dataset['val'])}"
print('Sanity check passed: dataset sizes look as expected.')

# can access instances like a regular Python list
print('example instance (dataset[\'train\'][8]):', dataset['train'][8])

Filter: 100%|██████████| 35748/35748 [00:00<00:00, 957874.30 examples/s]

train size: 147059
val size:   17874
Sanity check passed: dataset sizes look as expected.
example instance (dataset['train'][8]): {'text': 'The term "anatomy" is commonly taken to refer to human anatomy.  However, substantially the same structures and tissues are found throughout the rest of the animal kingdom and the term also includes the anatomy of other animals.  The term "zootomy" is also sometimes used to specifically refer to animals.  The structure and tissues of plants are of a dissimilar nature and they are studied in plant anatomy.'}


In [14]:
# Task 2.2 - Iterating through the datasets with DataLoader

from torch.utils.data import DataLoader

# keep this simple here (raw text batches), matching what the assignment asks for.
# Actual tokenization/collation for training happens later, once the tokenizer is ready.
BATCH_SIZE = 32
dl = DataLoader(dataset['train'], batch_size=BATCH_SIZE, shuffle=True)

# Sanity check: look at the first batch and confirm it matches expectations.
for batch in dl:
    print(batch)
    break

{'text': ['Ferries leave from Port Darwin to island locations, mainly for tourists.  A ferry service to the Tiwi Islands, the "Arafura Pearl" operates from Cullen Bay.', 'Tennis is also gaining a lot of popularity after the recent successes of Damir Džumhur and Mirza Bašić at Grand Slam level.  Other notable tennis players who represented Bosnia are, Amer Delić and Mervana Jugić-Salkić.', 'Above the capital is a square abacus connecting the capital to the entablature.  The Entablature is divided into three horizontal registers, the lower part of which is either smooth or divided by horizontal lines.  The upper half is distinctive for the Doric order.  The frieze of the Doric entablature is divided into triglyphs and metopes.  A triglyph is a unit consisting of three vertical bands which are separated by grooves.  Metopes are the plain or carved reliefs between two triglyphs.', "Towards the end of his career, it becomes clear that Poirot's retirement is no longer a convenient fiction.  

## AS2 Step 1: Setting up a Transformer neural network

In [15]:
# Part 3 - Transformer language model (Llama-style decoder-only)

import math
import torch
from torch import nn
import torch.nn.functional as F
from transformers import PreTrainedModel, PretrainedConfig


class A2ModelConfig(PretrainedConfig):
    """Stores all hyperparameters used to build the Transformer model."""
    def __init__(
        self,
        vocab_size=None,              # number of tokens in vocabulary
        hidden_size=None,             # model width (embedding/hidden dimension)
        intermediate_size=None,       # MLP expansion size
        num_attention_heads=None,     # number of attention heads
        num_hidden_layers=None,       # number of decoder blocks
        rope_theta=None,              # RoPE base frequency
        hidden_act='silu',            # activation for MLP (SwiGLU uses SiLU)
        max_position_embeddings=None, # max supported sequence length
        rms_norm_eps=None,            # epsilon for RMSNorm numerical stability
        **kwargs
    ):
        super().__init__(**kwargs)
        self.vocab_size = vocab_size
        self.hidden_size = hidden_size
        self.max_position_embeddings = max_position_embeddings
        self.rms_norm_eps = rms_norm_eps
        self.num_attention_heads = num_attention_heads
        self.rope_theta = rope_theta
        self.hidden_act = hidden_act
        self.intermediate_size = intermediate_size
        self.num_hidden_layers = num_hidden_layers


class A2MLP(nn.Module):
    """Transformer feed-forward block using SwiGLU."""
    def __init__(self, config):
        super().__init__()
        assert config.hidden_act == 'silu'  # expected by SwiGLU design

        # SwiGLU uses two parallel projections:
        # gate_proj -> SiLU(gate), up_proj -> value branch
        self.gate_proj = nn.Linear(config.hidden_size, config.intermediate_size, bias=False)
        self.up_proj = nn.Linear(config.hidden_size, config.intermediate_size, bias=False)

        # Project back to hidden size after elementwise gating
        self.down_proj = nn.Linear(config.intermediate_size, config.hidden_size, bias=False)

    def forward(self, hidden_states):
        # Output shape: (batch, seq_len, hidden_size)
        return self.down_proj(F.silu(self.gate_proj(hidden_states)) * self.up_proj(hidden_states))


class A2RMSNorm(nn.Module):
    """RMSNorm: normalize by root-mean-square, then scale by learnable weights."""
    def __init__(self, config):
        super().__init__()
        self.eps = config.rms_norm_eps
        self.weight = nn.Parameter(torch.ones(config.hidden_size))  # per-feature scale

    def forward(self, hidden_states):
        # Compute normalization in float32 for stability, then cast back.
        x = hidden_states.float()
        rms = torch.rsqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        x = x * rms
        return (self.weight * x).to(hidden_states.dtype)


class A2Attention(nn.Module):
    """The multi-head attention layer of the Transformer.
    Uses standard scaled dot-product attention with causal masking and RoPE.
    """

    def __init__(self, config):
        super().__init__()
        assert config.hidden_size % config.num_attention_heads == 0

        self.hidden_size = config.hidden_size
        self.num_heads = config.num_attention_heads
        self.head_dim = config.hidden_size // config.num_attention_heads

        # Linear projections for query, key, value, and output.
        self.W_q = nn.Linear(config.hidden_size, config.hidden_size, bias=False)
        self.W_k = nn.Linear(config.hidden_size, config.hidden_size, bias=False)
        self.W_v = nn.Linear(config.hidden_size, config.hidden_size, bias=False)
        self.W_o = nn.Linear(config.hidden_size, config.hidden_size, bias=False)

        # OLMo 2 also applies layer normalizers after the query and key representations.
        self.q_norm = A2RMSNorm(config)
        self.k_norm = A2RMSNorm(config)

    def forward(self, hidden_states, rope_rotations):
        bsz, seqlen, _ = hidden_states.shape

        # Compute query, key, value representations.
        q = self.W_q(hidden_states)
        k = self.W_k(hidden_states)
        v = self.W_v(hidden_states)

        # Apply normalization after the query and key projections.
        q = self.q_norm(q)
        k = self.k_norm(k)

        # Reshape so that individual attention heads are separated.
        q = q.view(bsz, seqlen, self.num_heads, self.head_dim).transpose(1, 2)  # (B, H, T, D)
        k = k.view(bsz, seqlen, self.num_heads, self.head_dim).transpose(1, 2)  # (B, H, T, D)
        v = v.view(bsz, seqlen, self.num_heads, self.head_dim).transpose(1, 2)  # (B, H, T, D)

        # Apply RoPE rotations to the query and key representations.
        q, k = apply_rotary_pos_emb(q, k, rope_rotations)

        # Compute attention pre-activations and scale them.
        attn_scores = torch.matmul(q, k.transpose(-1, -2)) / math.sqrt(self.head_dim)  # (B, H, T, T)

        # Add a causal mask so each token can only see earlier positions.
        causal_mask = torch.triu(
            torch.ones(seqlen, seqlen, device=hidden_states.device, dtype=torch.bool),
            diagonal=1
        )
        attn_scores = attn_scores.masked_fill(causal_mask, torch.finfo(attn_scores.dtype).min)

        # Convert scores to attention probabilities.
        attn_probs = torch.softmax(attn_scores, dim=-1)

        # Multiply the attention weights by the value tensor.
        attn_out = torch.matmul(attn_probs, v)  # (B, H, T, D)

        # Merge the heads back into the hidden size.
        attn_out = attn_out.transpose(1, 2).contiguous().view(bsz, seqlen, self.hidden_size)  # (B, T, C)

        # Final output projection.
        return self.W_o(attn_out)


class A2DecoderLayer(nn.Module):
    """A complete Transformer decoder layer."""
    def __init__(self, config):
        super().__init__()
        self.input_layernorm = A2RMSNorm(config)
        self.self_attn = A2Attention(config)
        self.post_attention_layernorm = A2RMSNorm(config)
        self.mlp = A2MLP(config)

    def forward(self, hidden_states, rope_rotations):
        # Residual connection around attention.
        hidden_states = hidden_states + self.self_attn(self.input_layernorm(hidden_states), rope_rotations)

        # Residual connection around MLP.
        hidden_states = hidden_states + self.mlp(self.post_attention_layernorm(hidden_states))
        return hidden_states


class A2Transformer(PreTrainedModel):
    """A language model based on the Transformer architecture."""
    config_class = A2ModelConfig

    def __init__(self, config):
        super().__init__(config)

        self.embedding = nn.Embedding(config.vocab_size, config.hidden_size)
        self.rotary_emb = A2RotaryEmbedding(config)
        self.layers = nn.ModuleList([A2DecoderLayer(config) for _ in range(config.num_hidden_layers)])
        self.final_norm = A2RMSNorm(config)
        self.unembedding = nn.Linear(config.hidden_size, config.vocab_size, bias=False)

        self.post_init()

    def forward(self, input_ids, labels=None):
        # Create RoPE rotations and pass them to each decoder layer.
        rope_rotations = self.rotary_emb(input_ids)

        hidden_states = self.embedding(input_ids)
        for layer in self.layers:
            hidden_states = layer(hidden_states, rope_rotations)

        hidden_states = self.final_norm(hidden_states)
        logits = self.unembedding(hidden_states)

        out = {"logits": logits}
        if labels is not None:
            # Standard causal LM loss:
            # predict token t+1 from hidden state at token t.
            shift_logits = logits[:, :-1, :].contiguous()   # (B, T-1, V)
            shift_labels = labels[:, 1:].contiguous()       # (B, T-1)

            # Flatten batch+time so CrossEntropy gets shape (N, V) vs (N,).
            # ignore_index=-100 skips padded target positions.
            loss = F.cross_entropy(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1),
                ignore_index=-100,
            )
            out["loss"] = loss
        return out


#### RoPE implementation (copied and simplified from HuggingFace).

def apply_rotary_pos_emb(q, k, rope_rotations, unsqueeze_dim=1):
    """Applies precomputed RoPE rotations to query/key tensors."""
    assert q.shape == k.shape
    assert len(q.shape) == 4  # expected: (B, H, T, D)

    # rope_rotations = (cos, sin), each shaped for (B, T, D)
    cos, sin = rope_rotations
    assert q.shape[2] == cos.shape[1]  # T matches
    assert q.shape[3] == cos.shape[2]  # D matches

    q_type, k_type = q.dtype, k.dtype

    # Add head dimension so broadcasting works with (B,H,T,D).
    cos = cos.unsqueeze(unsqueeze_dim)
    sin = sin.unsqueeze(unsqueeze_dim)

    # RoPE rotation formula:
    # x_rot = x * cos + rotate_half(x) * sin
    q_embed = (q * cos) + (rotate_half(q) * sin)
    k_embed = (k * cos) + (rotate_half(k) * sin)

    # Cast back to original dtypes.
    return q_embed.to(q_type), k_embed.to(k_type)


def rotate_half(x):
    """Rotates half the hidden dims of the input."""
    x1 = x[..., : x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2 :]
    return torch.cat((-x2, x1), dim=-1)


class A2RotaryEmbedding(nn.Module):
    """Builds cosine/sine RoPE factors for each sequence position."""
    def __init__(self, config, device=None):
        super().__init__()
        rope_theta = config.rope_theta

        # RoPE is applied per attention head dimension.
        head_dim = config.hidden_size // config.num_attention_heads
        partial_rotary_factor = 1.0  # full rotary on all head dims
        dim = int(head_dim * partial_rotary_factor)

        # Inverse frequencies for even positions in rotary dimension.
        self.inv_freq = 1.0 / (
            rope_theta ** (torch.arange(0, dim, 2, dtype=torch.int64).to(device=device, dtype=torch.float) / dim)
        )

    @torch.no_grad()
    def forward(self, x):
        # x is typically input_ids here; only batch size and seq length are used.
        position_ids = torch.arange(0, x.shape[1], device=x.device).unsqueeze(0)  # (1, T)

        # Expand inv_freq to match batch for batched matmul.
        inv_freq_expanded = self.inv_freq[None, :, None].float().expand(position_ids.shape[0], -1, 1).to(x.device)
        position_ids_expanded = position_ids[:, None, :].float()

        # Keep this computation in float32 for numerical stability.
        # On MPS, autocast handling is limited, so force cpu context string.
        device_type = x.device.type if isinstance(x.device.type, str) and x.device.type != "mps" else "cpu"
        with torch.autocast(device_type=device_type, enabled=False):
            # Outer product -> phase angles for each (position, frequency).
            freqs = (inv_freq_expanded.float() @ position_ids_expanded.float()).transpose(1, 2)  # (B, T, dim/2)

            # Duplicate so shape matches full rotary dimension dim.
            emb = torch.cat((freqs, freqs), dim=-1)  # (B, T, dim)

            # Return precomputed rotation tensors.
            cos = emb.cos()
            sin = emb.sin()
            return cos, sin


# Sanity check.
# Create a tiny input tensor and make sure the full stack returns logits
# with the correct vocabulary-size dimension.
if __name__ == "__main__":
    config = A2ModelConfig(
        vocab_size=1000,
        hidden_size=128,
        intermediate_size=512,
        num_attention_heads=8,
        num_hidden_layers=2,
        rope_theta=10000.0,
        hidden_act="silu",
        max_position_embeddings=256,
        rms_norm_eps=1e-6,
    )
    model = A2Transformer(config)
    x = torch.randint(0, config.vocab_size, (2, 16))
    out = model(x)
    assert out["logits"].shape == (2, 16, config.vocab_size)
    print("Sanity check passed.")

Sanity check passed.


## Training the model

In [16]:
# Part 4 - Training the Transformer model

import os
import math
import torch
from dataclasses import dataclass
from torch.utils.data import DataLoader

# 4.1) Build an A2 Transformer instance (this is the key AS2 switch from A1 RNN).
a2_config = A2ModelConfig(
    vocab_size=len(tokenizer),        # must match tokenizer vocabulary size
    hidden_size=256,                  # model width
    intermediate_size=1024,           # MLP inner size
    num_attention_heads=8,            # 256 / 8 = 32 dim per head
    num_hidden_layers=4,              # number of decoder blocks
    rope_theta=10000.0,               # standard RoPE base
    hidden_act="silu",                # needed by SwiGLU
    max_position_embeddings=256,      # same as tokenizer max length target
    rms_norm_eps=1e-6,                # RMSNorm stability epsilon
)
model = A2Transformer(a2_config).to(DEVICE)
print("A2 model ready on:", DEVICE)

A2 model ready on: mps


In [17]:
def make_collate(tokenizer):
    def collate(batch):
        texts = [ex["text"] for ex in batch]
        enc = tokenizer(texts, return_tensors="pt", padding=True, truncation=True)
        enc.pop("attention_mask", None)  # A2Transformer.forward doesn't use/accept this
        labels = enc["input_ids"].clone()
        labels[labels == tokenizer.pad_token_id] = -100
        enc["labels"] = labels
        return enc
    return collate

collate = make_collate(tokenizer)

# Quick sanity check
_test_batch = collate([{'text': 'This is a test.'}, {'text': 'Another one.'}])
print(_test_batch)
assert (_test_batch['labels'] == -100).sum().item() > 0 or _test_batch['input_ids'].shape[1] == \
       _test_batch['input_ids'].shape[1]  # just confirms it runs without crashing
print('collate() sanity check passed.')

{'input_ids': tensor([[  1,  35,  14,  11, 972,   6,   2],
        [  1, 155,  45,   6,   2,   0,   0]]), 'labels': tensor([[   1,   35,   14,   11,  972,    6,    2],
        [   1,  155,   45,    6,    2, -100, -100]])}
collate() sanity check passed.


In [18]:

# use a small learning rate and let the model update using the computed gradients.
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

# The validation loader is used to monitor generalization.
train_dl = DataLoader(
    dataset["train"],
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate,
)

eval_dl = DataLoader(
    dataset["val"],
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate,
)


for epoch in range(1):
    model.train()
    for step, batch in enumerate(train_dl):
        # Move every tensor in the batch to the active device (CPU / MPS / CUDA).
        batch = {k: v.to(DEVICE) for k, v in batch.items()}

        # Reset gradients before each batch.
        optimizer.zero_grad()

        # Run the model on the batch
        out = model(**batch)
        loss = out["loss"]

        # compute gradients of the loss 
        loss.backward()

        # Gradient clipping helps keep optimization stable
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        # Update the parameters using AdamW
        optimizer.step()

        # Print a few training losses for monitoring.
        if step % 50 == 0:
            print(f"epoch {epoch} step {step:5d} loss={loss.item():.4f}")

    # After finishing the epoch, evaluate on validation data.
    # The loss is computed using the same causal LM objective.
    model.eval()
    total_loss = 0.0
    total_tokens = 0

    with torch.no_grad():
        for batch in eval_dl:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            out = model(**batch)

            # Count only non-padding target tokens for the loss.
            n = (batch["labels"][:, 1:] != -100).sum().item()
            total_loss += out["loss"].item() * n
            total_tokens += n

    avg_loss = total_loss / max(1, total_tokens)
    print(f"== epoch {epoch}: eval_loss={avg_loss:.4f} eval_ppl={math.exp(avg_loss):.2f}")

# Save the trained model and tokenizer so I can reload them later if needed.
# These are useful artifacts for assignment reporting and debugging.
os.makedirs("a2_transformer_model", exist_ok=True)
torch.save(
    {
        "model_state_dict": model.state_dict(),
        "config": model.config.to_dict(),
    },
    os.path.join("a2_transformer_model", "model.pt"),
)

tokenizer.save(os.path.join("a2_transformer_model", "tokenizer.json"))
print("Saved model and tokenizer to a2_transformer_model")

epoch 0 step     0 loss=9.2675
epoch 0 step    50 loss=5.9795
epoch 0 step   100 loss=5.8062
epoch 0 step   150 loss=5.4498
epoch 0 step   200 loss=5.4617
epoch 0 step   250 loss=5.3015
epoch 0 step   300 loss=5.3768
epoch 0 step   350 loss=5.2694
epoch 0 step   400 loss=5.3848
epoch 0 step   450 loss=5.1410
epoch 0 step   500 loss=5.0960
epoch 0 step   550 loss=4.8836
epoch 0 step   600 loss=5.0487
epoch 0 step   650 loss=5.0351
epoch 0 step   700 loss=4.8632
epoch 0 step   750 loss=4.7602
epoch 0 step   800 loss=5.0334
epoch 0 step   850 loss=4.8366
epoch 0 step   900 loss=4.8859
epoch 0 step   950 loss=4.7979
epoch 0 step  1000 loss=4.8805
epoch 0 step  1050 loss=4.7619
epoch 0 step  1100 loss=4.6838
epoch 0 step  1150 loss=4.6091
epoch 0 step  1200 loss=4.6904
epoch 0 step  1250 loss=4.6676
epoch 0 step  1300 loss=4.7369
epoch 0 step  1350 loss=4.7266
epoch 0 step  1400 loss=4.4840
epoch 0 step  1450 loss=4.4679
epoch 0 step  1500 loss=4.6792
epoch 0 step  1550 loss=4.5181
epoch 0 

## Part 3 of AS2 — Generating text

In [19]:
# Task 3.1 — Predicting the next word

import torch

model.eval()

@torch.no_grad()
def predict_next_word(model, tokenizer, text):
    enc = tokenizer([text], return_tensors="pt")
    input_ids = enc["input_ids"].to(DEVICE)
    input_ids = input_ids[:, :-1]  # remove EOS dummy before predicting continuation
    out = model(input_ids=input_ids)

    # take logits at the last position and choose the highest-scoring token
    logits = out["logits"][0, -1]
    next_token_id = torch.argmax(logits, dim=-1).item()
    return tokenizer.itos[next_token_id]

for text in [
    "She lives in San",
    "The president of the united",
    "Water boils at a temperature of",
]:
    pred = predict_next_word(model, tokenizer, text)
    print(f"{text!r} -> {pred!r}")

'She lives in San' -> 'salvador'
'The president of the united' -> 'states'
'Water boils at a temperature of' -> '<unk>'


In [21]:
# Task 3.2 — Generating texts

import torch
from torch.distributions import Categorical

@torch.no_grad()
def generate_text(model, tokenizer, prompt, max_length=50, temperature=1.0, topk=None):
    model.eval()

    input_ids = tokenizer([prompt], return_tensors="pt")["input_ids"].to(DEVICE)

    for _ in range(max_length):
        out = model(input_ids=input_ids)
        logits = out["logits"][:, -1, :] / temperature

        if topk is not None:
            topk_vals, topk_idx = torch.topk(logits, k=min(topk, logits.size(-1)))
            filtered = torch.full_like(logits, float("-inf"))
            filtered.scatter_(1, topk_idx, topk_vals)
            logits = filtered

        probs = torch.softmax(logits, dim=-1)
        next_token = Categorical(probs=probs).sample()
        input_ids = torch.cat([input_ids, next_token.unsqueeze(1)], dim=1)

        if next_token.item() == tokenizer.eos_token_id:
            break

    tokens = tokenizer.decode(input_ids[0].tolist())
    cleaned = []
    for t in tokens:
        if t in {tokenizer.pad_token, tokenizer.bos_token, tokenizer.eos_token}:
            continue
        if t == tokenizer.unk_token:
            t = "[UNK]"
        cleaned.append(t)

    return " ".join(cleaned).strip()

for prompt in [
    "In natural language processing, a Transformer",
    "Is Stockholm the capital of Sweden? Answer yes or no. The answer is",
    "Write a Python program that reverses a list.",
]:
    print(f"\nPROMPT: {prompt}")
    print("GENERATED:")
    print(generate_text(model, tokenizer, prompt, max_length=30, temperature=0.8, topk=20))


PROMPT: In natural language processing, a Transformer
GENERATED:
in natural language processing , a [UNK] , or a [UNK] dialect , a form of [UNK] . these two are [UNK] , [UNK] ( `` [UNK] [UNK] '' , `` [UNK] [UNK] [UNK] '' ) .

PROMPT: Is Stockholm the capital of Sweden? Answer yes or no. The answer is
GENERATED:
is stockholm the capital of sweden ? answer yes or no . the answer is , and the average [UNK] of the island is [UNK] ( a [UNK] ) .

PROMPT: Write a Python program that reverses a list.
GENERATED:
write a [UNK] program that [UNK] a list . authors can find a [UNK] 's degree of [UNK] , for example , in which a number of words is [UNK] ( e.g. , `` , i 'm , ''


In [22]:
# Task 3.3 — Comparing to a pretrained Transformer baseline via Ollama

# If the model is not installed locally yet, run this in the terminal:
# ollama pull mistral:7b

try:
    import ollama
except ModuleNotFoundError as e:
    raise RuntimeError(
        "Ollama Python package is missing. Install with:\n"
        "python -m pip install ollama"
    ) from e

MODEL_NAME = "mistral:7b"

def generate_ollama_text(prompt, model_name=MODEL_NAME, max_new_tokens=40, temperature=0.8):
    response = ollama.generate(
        model=model_name,
        prompt=prompt,
        options={
            "temperature": temperature,
            "top_k": 20,
            "num_predict": max_new_tokens,
        },
    )
    return response["response"]

for prompt in [
    "In natural language processing, a Transformer",
    "Is Stockholm the capital of Sweden? Answer yes or no. The answer is",
    "Write a Python program that reverses a list.",
]:
    print("\nPROMPT:", prompt)
    print("Pretrained Ollama model:", MODEL_NAME)
    print(generate_ollama_text(prompt, model_name=MODEL_NAME, max_new_tokens=40, temperature=0.8))


PROMPT: In natural language processing, a Transformer
Pretrained Ollama model: mistral:7b
 In natural language processing, a Transformer is a type of model architecture introduced by Vaswani et al. in the paper "Attention is All You Need". It's a model architecture for

PROMPT: Is Stockholm the capital of Sweden? Answer yes or no. The answer is
Pretrained Ollama model: mistral:7b
 Yes

PROMPT: Write a Python program that reverses a list.
Pretrained Ollama model: mistral:7b
 Here is a simple Python program that reverses a list:

```python
# Function to reverse a list
def reverse_list(input_list):
    return input_list


## my reflection:
The pretrained model (mistral:7b) is answering correctly to the questions while the one that we trained in this homework answered wrongly to the prompts. For instance in one case it just repeat the question, and to none of the question it could reply correctly.